## 1. Imports

In [1]:
import numpy
import karateclub as kc
import matplotlib.pyplot as plt
import networkx as nx
from networkx.drawing.nx_pydot import graphviz_layout
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings

from notebook_utils import compute_extended_explore_traj, compute_explore_traj
from flow_tree_utils import generate_trajs_from_df, generate_flow_tree_from_trajs, plot_flow_tree
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score


warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

## 2. Load data

In [2]:
df = pd.read_csv("data/MLINDIV_train_full.csv")

### Load test df

In [3]:
# load test df
test_df = df[df["eprocs"].str.contains("Test")]

test_df["subj_mean_acc"] = test_df.groupby("Subject")["accuracy"].transform("mean")
test_df["path_mean_acc"] = test_df.groupby(["StartAt", "EndAt"])["accuracy"].transform("mean")

test_df['quartile'] = (
    pd.qcut(test_df['subj_mean_acc'], 4, labels=[1, 2, 3, 4])
)

test_df["path_mean_acc_by_Q"] = test_df.groupby(["StartAt", "EndAt", "quartile"])["accuracy"].transform("mean")

### Load explore df

In [4]:
# load explore df
explore_df = df[df["eprocs"].str.contains("Explore")]

# By quartile
q_dict = {}
acc_dict = {}
for index, row in test_df[["Subject", "quartile", "subj_mean_acc"]].iterrows():
    subj, q, acc = row["Subject"], row["quartile"], row["subj_mean_acc"]
    if subj not in q_dict:
        q_dict[subj] = q
        acc_dict[subj] = acc

explore_df["quartile"] = explore_df["Subject"].map(q_dict)
explore_df["subject_mean_acc"] = explore_df["Subject"].map(acc_dict)

# concatenate explore rounds for now
explore_df["explore_path"] = explore_df.groupby("Subject")["e_paths"].transform(" ".join)
explore_df = explore_df.drop_duplicates(subset=["Subject"], keep="first")

explore_df["extended_explore_traj"] = explore_df["explore_path"].apply(compute_extended_explore_traj)
explore_df["explore_traj"] = explore_df["explore_path"].apply(compute_explore_traj)
explore_df["num_nodes_visited"] = explore_df["explore_traj"].apply(len)

### Load subject df

In [5]:
subject_df = pd.read_csv("data/MLINDIV_subject_info.csv")
subject_df["Subject"] = pd.to_numeric(subject_df["Spatial Neuro ID"], errors="coerce")
subject_df = subject_df[~subject_df["Subject"].isna()]

subject_df["is_man"] = (subject_df["Sex"] == "M")

merged_df = pd.merge(explore_df, subject_df, on="Subject")
# merged_df[["Sex", "Proportion Correct (new version)", "subject_mean_acc"]]

## 3. Set up how to run hypothesis test

1. Compute Flow Trees
2. Compute features on Flow Trees
3. Run appropriate statistical test


In [272]:
from scipy.spatial.distance import pdist
from scipy import stats 


def vector_statistic(x, y, axis):
    mean_x = np.mean(x, axis=axis)
    mean_y = np.mean(y, axis=axis)
        
    # Calculate Euclidean distance
    dist = mean_x - mean_y
    # print(f"Distance between means: {dist}")  # Debug print

    return dist

def run_ttest(group1_reps, group2_reps, statistic=vector_statistic):
    # Perform a t-test on groups
    t_stat, p_value = stats.ttest_ind(group1_reps, group2_reps)
    print(f"Group 1 mean: {np.mean(group1_reps):.2f} (var={np.var(group1_reps):.2f}; n={len(group1_reps)})")
    print(f"Group 2 mean: {np.mean(group2_reps):.2f} (var={np.var(group2_reps):.2f}; n={len(group2_reps)})")
    print(f"T-statistic: {t_stat:.4f}, P-value: {p_value:.4f}")

    # mean1, var1, n1 = np.mean(group1_reps), np.var(group1_reps), len(group1_reps)
    # mean2, var2, n2 = np.mean(group2_reps), np.var(group2_reps), len(group2_reps)
    # t = (mean1 - mean2) / np.sqrt((var1/n1 + var2/n2))
    # print("Number samples:", n1, n2)
    # print("T homemade:", t)

    return t_stat, p_value


def run_permutation_test(group1_reps, group2_reps, statistic=vector_statistic):
    res = stats.permutation_test((group1_reps, group2_reps), statistic, vectorized=True,
                       n_resamples=9999,# np.inf,
                       alternative='two-sided')
    print(f"Permutation test result: {res.statistic:.4} (p={res.pvalue:.4})")

    if len(group1_reps.shape) == 1:
        group1_reps = group1_reps.reshape(-1, 1)
    if len(group2_reps.shape) == 1:
        group2_reps = group2_reps.reshape(-1, 1)

    all_group_dist = pdist(np.concatenate([group1_reps, group2_reps]))

    within_group1_dist = pdist(group1_reps)
    within_group2_dist = pdist(group2_reps)
    within_group_dist = np.concatenate([within_group1_dist, within_group2_dist])

    print(f"Mean within group 1 dist: {np.mean(within_group1_dist):.2} (var={np.var(within_group1_dist):.2})")
    print(f"Mean within group 2 dist: {np.mean(within_group2_dist):.2} (var={np.var(within_group2_dist):.2})")
    print(f"Mean between all group dist: {np.mean(all_group_dist):.2} (var={np.var(all_group_dist):.2})")

    res2 = stats.permutation_test((within_group1_dist, all_group_dist), statistic, vectorized=True,
                       n_resamples=9999,#np.inf, 
                       alternative='less') # within group dist should be less than between
    print(f"Within group 1 dist vs among all: {res2.statistic:.4} (p={res2.pvalue:.4})")  
    t_stat, p_value = stats.ttest_ind(within_group1_dist, all_group_dist)
    print(f"\tT-statistic: {t_stat:.4f}, P-value: {p_value:.4f}")  

    res2 = stats.permutation_test((within_group2_dist, all_group_dist), statistic, vectorized=True,
                       n_resamples=9999,#np.inf, 
                       alternative='less') # within group dist should be less than between
    print(f"Within group 2 dist vs among all: {res2.statistic:.4} (p={res2.pvalue:.4})")  
    t_stat, p_value = stats.ttest_ind(within_group2_dist, all_group_dist)
    print(f"\tT-statistic: {t_stat:.4f}, P-value: {p_value:.4f}")  

    res2 = stats.permutation_test((within_group1_dist, within_group2_dist), statistic, vectorized=True,
                       n_resamples=9999,#np.inf, 
                       alternative='less') # within group dist should be less than between
    print(f"Within group 1 dist vs within group 2: {res2.statistic:.4} (p={res2.pvalue:.4})")  
    t_stat, p_value = stats.ttest_ind(within_group1_dist, within_group2_dist)
    print(f"\tT-statistic: {t_stat:.4f}, P-value: {p_value:.4f}")   

def euc_dist_between_means(x, y):
    return np.linalg.norm(np.mean(x, axis=0) - np.mean(y, axis=0))


def run_homemade_permutation_test(group1, group2, statistic=euc_dist_between_means, n_permutations=999):
    observed = statistic(group1, group2)

    data = np.concatenate([group1, group2])
    rng = np.random.default_rng()

    n_samples = len(data)

    n_group1 = len(group1)

    perm_generator = (rng.permutation(n_samples)
                            for i in range(n_permutations))

    def _batch_generator(iterable, batch):
        """A generator that yields batches of elements from an iterable"""
        iterator = iter(iterable)
        if batch <= 0:
            raise ValueError("`batch` must be positive.")
        z = [item for i, item in zip(range(batch), iterator)]
        while z:  # we don't want StopIteration without yielding an empty list
            yield z
            z = [item for i, item in zip(range(batch), iterator)]


    null_distribution = []
    for indices in _batch_generator(perm_generator, batch=n_permutations):
        indices = np.array(indices) # shape: [num permutations, num samples]

        for permutation in indices:
            data_batch = data[permutation, ...]

            permuted_group1 = data_batch[:n_group1, ...]
            permuted_group2 = data_batch[n_group1:, ...]

            permuted_statistic = statistic(permuted_group1, permuted_group2)
            null_distribution.append(permuted_statistic)

    null_distribution = np.array(null_distribution)

    adjustment = 1
    n_resamples = n_permutations

    def less(null_distribution, observed):
        cmps = null_distribution <= observed #+ gamma
        pvalues = (cmps.sum(axis=0) + adjustment) / (n_resamples + adjustment)
        return pvalues

    def greater(null_distribution, observed):
        cmps = null_distribution >= observed #- gamma
        pvalues = (cmps.sum(axis=0) + adjustment) / (n_resamples + adjustment)
        return pvalues

    def two_sided(null_distribution, observed):
        pvalues_less = less(null_distribution, observed)
        pvalues_greater = greater(null_distribution, observed)
        pvalues = np.minimum(pvalues_less, pvalues_greater) * 2
        return pvalues

    pvalue = two_sided(null_distribution, observed)
    print(f"Permutation test: {observed:.4f} (p={pvalue:.4f})")

    return observed, pvalue

## 4. Run hypothesis tests

Hypotheses to test:

1. Duh -- Do quartiles have different group dynamics?
2. Are group dynamics of reversed flow trees different? -- need to bootstrap??


In [275]:
# QUARTILES DIFFERENT
q12_flow_trees = []
q34_flow_trees = []

for (start_node, end_node, quartile), group in test_df.groupby(["StartAt", "EndAt", "quartile"]):
    trajs = generate_trajs_from_df(group)
    G = generate_flow_tree_from_trajs(trajs, END_NODE=end_node)
    
    # for embeddings to work
    new_G = nx.relabel.convert_node_labels_to_integers(G, first_label=0, ordering='default') #first_label is the starting integer label, in this case zero

    if quartile <= 2:
        q12_flow_trees.append(new_G)
    else:
        q34_flow_trees.append(new_G)


print("Do Flow Trees from top and bottom half of performers have different diameters?")
q12_diams = np.array([nx.diameter(FT) for FT in q12_flow_trees])
q34_diams = np.array([nx.diameter(FT) for FT in q34_flow_trees])

run_ttest(q12_diams, q34_diams)
print()
run_permutation_test(q12_diams, q34_diams)
print()
run_homemade_permutation_test(q12_diams, q34_diams)


Do Flow Trees from top and bottom half of performers have different diameters?
Group 1 mean: 8.17 (var=1.92; n=144)
Group 2 mean: 4.15 (var=4.31; n=144)
T-statistic: 19.3026, P-value: 0.0000

Permutation test result: 4.028 (p=0.0002)
Mean within group 1 dist: 1.5 (var=1.5)
Mean within group 2 dist: 2.4 (var=3.0)
Mean between all group dist: 3.0 (var=5.1)
Within group 1 dist vs among all: -1.513 (p=0.0001)
	T-statistic: -65.6464, P-value: 0.0000
Within group 2 dist vs among all: -0.6748 (p=0.0001)
	T-statistic: -28.3059, P-value: 0.0000
Within group 1 dist vs within group 2: -0.8386 (p=0.0001)
	T-statistic: -39.8921, P-value: 0.0000

Permutation test: 4.0278 (p=0.0020)


(4.027777777777778, 0.002)

In [277]:
print("Do Flow Trees from top and bottom half of performers have different embeddings?")
test_flow_tree_embedder = kc.Graph2Vec(dimensions=64)
test_flow_tree_embedder.fit(q12_flow_trees + q34_flow_trees)

q12_embeddings = test_flow_tree_embedder.infer(q12_flow_trees)
q34_embeddings = test_flow_tree_embedder.infer(q34_flow_trees)

q12_embedding_means = np.array([100*np.mean(x) for x in q12_embeddings])
q34_embedding_means = np.array([100*np.mean(x) for x in q34_embeddings])

run_ttest(q12_embedding_means, q34_embedding_means)
print()
run_permutation_test(q12_embedding_means, q34_embedding_means)
print()
run_homemade_permutation_test(q12_embeddings, q34_embeddings)


Do Flow Trees from top and bottom half of performers have different embeddings?
Group 1 mean: 0.45 (var=0.01; n=144)
Group 2 mean: 0.22 (var=0.02; n=144)
T-statistic: 14.7426, P-value: 0.0000

Permutation test result: 0.2238 (p=0.0002)
Mean within group 1 dist: 0.14 (var=0.011)
Mean within group 2 dist: 0.15 (var=0.013)
Mean between all group dist: 0.2 (var=0.02)
Within group 1 dist vs among all: -0.05803 (p=0.0001)
	T-statistic: -39.4419, P-value: 0.0000
Within group 2 dist vs among all: -0.04341 (p=0.0001)
	T-statistic: -29.1580, P-value: 0.0000
Within group 1 dist vs within group 2: -0.01461 (p=0.0001)
	T-statistic: -9.6170, P-value: 0.0000

Permutation test: 0.3298 (p=0.0020)


(0.32982612, 0.002)

In [278]:
tree_dict = {}
winner_tree_dict = {}
loser_tree_dict = {}
for (start_node, end_node), group in test_df.groupby(["StartAt", "EndAt"]):
    trajs = generate_trajs_from_df(group)
    G = generate_flow_tree_from_trajs(trajs, END_NODE=end_node)

    new_G = nx.relabel.convert_node_labels_to_integers(G, first_label=0, ordering='default') # need to do for embeddings
    tree_dict[(start_node, end_node)] = [new_G, group["path_mean_acc"].iloc[0]]

    group["accuracy"] = group["accuracy"].astype(bool)

    # WINNERS ONLY
    trajs = generate_trajs_from_df(group[group["accuracy"]])
    G = generate_flow_tree_from_trajs(trajs, END_NODE=end_node)
    winner_tree_dict[(start_node, end_node)] = G

    # LOSERS ONLY
    trajs = generate_trajs_from_df(group[~group["accuracy"]])
    G = generate_flow_tree_from_trajs(trajs, END_NODE=end_node)
    loser_tree_dict[(start_node, end_node)] = G

In [279]:
# There and back
OBJECTS = "AIKLNOPWY"

acc_diffs = []
obj_pairs = []
for i in range(len(OBJECTS) - 1):
    for j in range(i+1, len(OBJECTS)):
        object_a, object_b = OBJECTS[i], OBJECTS[j] 

        acc_diffs.append(np.abs(tree_dict[(object_a, object_b)][1] - tree_dict[(object_b, object_a)][1]))
        obj_pairs.append((object_a, object_b))
        

pairs_with_diff = []
pairs_no_diff = []

accs_with_diff = []
accs_no_diff = []

sorted_order = [x for x in sorted(range(len(acc_diffs)), key=lambda x: acc_diffs[x])]
for i, tree_index in enumerate(sorted_order):
    node1, node2 = obj_pairs[tree_index]
    print(f"{node1}, {node2} acc diff is {acc_diffs[tree_index]}")

    test_df["accuracy"] = test_df["accuracy"].astype(bool)
    df1 = test_df[(test_df["StartAt"] == node1) & (test_df["EndAt"] == node2) & test_df["accuracy"]]
    df2 = test_df[(test_df["StartAt"] == node2) & (test_df["EndAt"] == node1) & test_df["accuracy"]]

    LA_group = []
    AL_group = []

    for i in range(30):
        # print(test_df[(test_df["StartAt"] == node1) & (test_df["EndAt"] == node2)])
        sample_df1 = df1.sample(n=10, random_state=i) 
        sample_df2 = df2.sample(n=10, random_state=i) 

        trajs1 = generate_trajs_from_df(sample_df1)
        G1 = generate_flow_tree_from_trajs(trajs1, END_NODE=node2)

        trajs2 = generate_trajs_from_df(sample_df2)
        G2 = generate_flow_tree_from_trajs(trajs2, END_NODE=node1)

        LA_group.append(G1)
        AL_group.append(G2)

    # for embeddings to work
    # new_G = nx.relabel.convert_node_labels_to_integers(G, first_label=0, ordering='default') #first_label is the starting integer label, in this case zero


    print("Do Flow Trees from top and bottom half of performers have different diameters?")
    group1_diams = [nx.diameter(FT) for FT in LA_group]
    group2_diams = [nx.diameter(FT) for FT in AL_group]

    t_stat, p_value = stats.ttest_ind(group1_diams, group2_diams)
    if p_value < 0.05:
        print("yes")
        pairs_with_diff.append((node1, node2))
        accs_with_diff.append(acc_diffs[tree_index])
    else:
        print("no")
        pairs_no_diff.append((node1, node2))
        accs_no_diff.append(acc_diffs[tree_index])

        
    print(f"T-statistic: {t_stat:.3}, P-value: {p_value:.3}")
    run_homemade_permutation_test(group1_diams, group2_diams)
    print()

    

print("With diff", np.mean(accs_with_diff), len(pairs_with_diff), pairs_with_diff)
print("No diff", np.mean(accs_no_diff), len(pairs_no_diff), pairs_no_diff)

K, L acc diff is 0.008215962441314506
Do Flow Trees from top and bottom half of performers have different diameters?
yes
T-statistic: -2.9, P-value: 0.00533
Permutation test: 0.6333 (p=0.0160)

K, Y acc diff is 0.014260249554367221
Do Flow Trees from top and bottom half of performers have different diameters?
yes
T-statistic: -8.36, P-value: 1.51e-11
Permutation test: 1.8667 (p=0.0020)

L, Y acc diff is 0.014285714285714235
Do Flow Trees from top and bottom half of performers have different diameters?
yes
T-statistic: -4.64, P-value: 2.01e-05
Permutation test: 1.2333 (p=0.0020)

A, K acc diff is 0.015597147950089152
Do Flow Trees from top and bottom half of performers have different diameters?
yes
T-statistic: 6.36, P-value: 3.46e-08
Permutation test: 1.1000 (p=0.0020)

W, Y acc diff is 0.022913601009039275
Do Flow Trees from top and bottom half of performers have different diameters?
no
T-statistic: -0.374, P-value: 0.709
Permutation test: 0.1000 (p=0.7860)

A, I acc diff is 0.0239898